# 12b - Inferential statistics (flips verification invariant #10 FAIL -> PASS)

> Copyright (C) 2024-2026 Marco Heinzen - SPDX-License-Identifier: AGPL-3.0-or-later
> Part of the Master Thesis "Building Damage Assessment with Multimodal Satellite Time Series and Machine Learning in the Russia-Ukraine War 2022-2026"
> Code hosted at https://github.com/marcoheinzen/bda
> Parts of this code were written or improved with the assistance of Claude (Anthropic); all other code, and the concept, research, architecture, design, execution, testing and validation throughout, are the author's work.


Runs on the **per-city AUC matrix from 12a** (run 12a first). CV is leave-one-city-out, so cities are the matched
blocks. Provides:
- **Bootstrap BCa CIs** on each experiment's mean-of-folds (resampling cities)
- **Friedman + Holm** omnibus across a selected experiment set (complete cities)
- **Wilcoxon signed-rank** pairwise (paired across cities) with Holm correction
- **Cliff's delta** effect sizes
- (optional) city-block bootstrap CI on **pooled** AUC (reloads OOF)

**No DeLong** - its i.i.d. assumption is violated under spatial autocorrelation. Run in WSL/JupyterLab.

In [1]:
import sys, re
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import bootstrap, friedmanchisquare, wilcoxon

_known = [Path("/content/drive_f/masterthesis/notebooks"),
          Path("/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/notebooks"),
          Path(r"F:\PROJECTS\masterthesis\gdrive\masterthesis\notebooks")]
_nb = next((c for c in list(Path.cwd().parents) + _known if (c / "global_setup.py").exists()), None)
if _nb is None: raise RuntimeError("global_setup.py not found")
if str(_nb) not in sys.path: sys.path.insert(0, str(_nb))
import global_setup as gs

RESULTS_ROOT = Path(gs.RESULTS_ROOT)
NB12 = RESULTS_ROOT / "nb12"
mat = pd.read_csv(NB12 / "nb12a_per_city_auc_matrix.csv", index_col=0)   # rows=experiment, cols=city
summ = pd.read_csv(NB12 / "nb12a_meanfolds_vs_pooled.csv")
print("matrix:", mat.shape, "(experiments x cities)")
print("experiments available:", len(mat))

def cliffs_delta(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if len(x)==0 or len(y)==0: return np.nan
    gt = sum(int((xi > y).sum()) for xi in x)
    lt = sum(int((xi < y).sum()) for xi in x)
    return (gt - lt) / (len(x)*len(y))

def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(order):
        val = (m - rank) * p[idx]
        run = max(run, val); adj[idx] = min(run, 1.0)
    return adj

/home/alpineobotics/miniconda3/envs/bda/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


BDA GLOBAL SETUP
Started: 2026-07-03 08:18:22
Python: 3.12.12

[1/7] Directory Structure
----------------------------------------------------------------------
  GDrive (G:):       /content/drive_f/masterthesis OK
  GDrive (F:):       /content/drive_f/masterthesis OK
  Local data (G:):   /content/masterthesis_local/data OK
  Data stack (F:):   /mnt/f/PROJECTS/masterthesis/data_stack OK

  TIER_SELECTION: [0, 1, 2]
  CITY_SELECTION: None (tier filter)
  REQUIRE_UNOSAT: False
  CITIES_TO_PROCESS: 21 cities

[2/7] Credentials
----------------------------------------------------------------------
  Copernicus: inf***
  OpenTopography: OK
  Earthdata: marcoheinzen

[3/7] Python Packages
----------------------------------------------------------------------


/content/drive_f/masterthesis/notebooks/global_setup.py:564: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



  Already installed: 23
  Newly installed:   0
  Failed:            0

[4/7] Global Imports & Configuration
----------------------------------------------------------------------
  All imports loaded

[5/7] Processing Config & SNAP
----------------------------------------------------------------------
  GPT: Usage:
  Temporal baseline: 10-24 days
  Wavelength: 0.0555

[6/7] GPU Status
----------------------------------------------------------------------
  CUDA available: NVIDIA GeForce RTX 2070 SUPER
    CUDA version: 12.8

[7/7] Disk Space
----------------------------------------------------------------------
  GDrive (G:)     930.7/7452.0 GB (6521.3 GB free)
  GDrive (F:)     1405.7/3726.0 GB (2320.3 GB free)
  Local data      11563.4/14901.9 GB (3338.4 GB free)
  Data stack      1405.7/3726.0 GB (2320.3 GB free)
  WSL ext4        69.1/1006.9 GB (886.6 GB free)

GLOBAL SETUP COMPLETE
  Torch device: cuda
  Cities: 21, CITY=Avdiivka
  Functions: load_aoi(), load_aoi_gdf(), load_aoi_

In [2]:
from pathlib import Path
import global_setup as gs
print("DRIVE_ROOT  :", gs.DRIVE_ROOT)
print("RESULTS_ROOT:", gs.RESULTS_ROOT)
for p in [Path(gs.RESULTS_ROOT) / "nb12",
          Path("/content/drive_f/masterthesis/results/nb12"),
          Path("/mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/results/nb12")]:
    f = p / "nb12a_per_city_auc_matrix.csv"
    print(f.exists(), str(f))
    

DRIVE_ROOT  : /content/drive_f/masterthesis
RESULTS_ROOT: /content/drive_f/masterthesis/results
True /content/drive_f/masterthesis/results/nb12/nb12a_per_city_auc_matrix.csv
True /content/drive_f/masterthesis/results/nb12/nb12a_per_city_auc_matrix.csv
True /mnt/f/PROJECTS/masterthesis/gdrive/masterthesis/results/nb12/nb12a_per_city_auc_matrix.csv


## Build the comparison sets (best-per-modality, best-per-classifier, dietrich28, top-10)

The earlier `SELECT=None` fallback compared the top-K by mean-folds, which were six near-sibling RGB models - a
low-information omnibus. Instead, tag every experiment in the 12a matrix by modality and classifier and build four
comparison sets:

- `best_per_modality` - one best experiment per sensor regime (the cross-approach test)
- `best_per_classifier_anyFS` - one best per classifier across any feature set (DESCRIPTIVE only: confounds
  classifier with feature set)
- `dietrich28_classifier_fixed` - classifiers on the fixed dietrich28 / block_stats feature set (the clean "does
  classifier choice matter after tuning" test)
- `top10` - top 10 by mean-folds (leaderboard distinguishability)

Each set is a SEPARATE Friedman comparison (mixing sets into one omnibus is uninterpretable). Friedman needs cities
scored in EVERY member of a set, so the cell prints the jointly-complete city count per set; where that is small,
lean on the BCa CIs (no jointness needed) and the pairwise Wilcoxon (each pair on its own complete cities). Edit
`MIN_CITIES` or the set definitions to match the exact experiments you report.

In [3]:
# Build candidate comparison sets from the 12a matrix, with complete-city diagnostics.
# Each set is a SEPARATE Friedman comparison; BCa CIs run over the union (no jointness needed).
MIN_CITIES = 8   # ignore experiments scored on fewer than this when picking a "best"

mf = summ.set_index("experiment")["mean_folds_auc"]

def modality_of(sig):
    s = str(sig).lower()
    nb11 = {
        "card_drop":"SAR_card_only", "prepost_single_card":"SAR_card_only",
        "coh_drop":"SAR_coh_only", "rolling_accum_coh":"SAR_coh_only",
        "fusion_card_cohdrop":"SAR_card+coh",
        "fusion_ms_card_cohdrop":"multimodal_all", "fusion_indices_card_cohdrop":"multimodal_all",
        "fusion_ms_cohdrop":"multimodal_SAR+MS", "fusion_composite_cohdrop":"multimodal_all",
        "fusion_composite_blockstats":"multimodal_all", "block_stats":"multimodal_all",
        "rolling_stats_roll3":"multimodal_all", "rolling_stats_roll7":"multimodal_all",
        "rolling_stats_roll13":"multimodal_all",
        "ms_change":"MS_only", "composite_prepost_bands":"MS_only", "composite_vs_scenes_bands":"MS_only",
        "lu_change":"other", "composite_prepost_landuse":"other", "composite_vs_scenes_landuse":"other",
    }
    if "nb11" in s:
        for k, m in nb11.items():
            if k in s: return m
    if any(x in s for x in ["coh_drop", "coh_running", "coh_max_drop"]): return "SAR_coh_unsupervised"
    if "logratio" in s:
        return "MS_unsupervised" if any(x in s for x in ["ndvi", "savi", "bsi", "nbr", "ndbi"]) else "SAR_unsupervised"
    if "card_only" in s: return "SAR_card_only"
    if "coh_only" in s: return "SAR_coh_only"
    if "card+coh" in s: return "SAR_card+coh"
    if "ms_only" in s: return "MS_only"
    if "nb08b" in s and "rgb" in s:
        return "MS_RGB+SAR" if ("card" in s or "coh" in s) else "MS_RGB_singlescene"
    if "all_multimodal" in s: return "multimodal_all"
    return "other"

def clf_of(sig):
    s = str(sig).lower()
    for name, pat in [("LightGBM-MIA","lightgbm-mia"), ("HistGBM","histgbm"), ("XGBoost","xgboost"),
                      ("ExtraTrees","extratrees"), ("RF-200","rf-200"), ("RF","rf_optuna"),
                      ("AdaBoost","adaboost"), ("LogReg","logreg"), ("Voting","voting"),
                      ("LightGBM","lightgbm"), ("GBM","gbm")]:
        if pat in s: return name
    return "other"

def ncities(sig):
    return int(mat.loc[sig].notna().sum()) if sig in mat.index else 0
def complete_cities(members):
    members = [m for m in members if m in mat.index]
    return mat.loc[members].dropna(axis=1, how="any").shape[1] if members else 0

eligible = [s for s in mat.index if s in mf.index and mf[s] == mf[s] and ncities(s) >= MIN_CITIES]
tag = pd.DataFrame({"sig": eligible})
tag["mean_folds"] = tag["sig"].map(mf)
tag["n_cities"]   = tag["sig"].map(ncities)
tag["modality"]   = tag["sig"].map(modality_of)
tag["clf"]        = tag["sig"].map(clf_of)

def best_per(col):
    return (tag.sort_values("mean_folds", ascending=False)
               .drop_duplicates(col).sort_values("mean_folds", ascending=False))

SET_modality   = best_per("modality")
SET_clf_anyfs  = best_per("clf")
SET_dietrich28 = tag[tag["sig"].str.contains("dietrich28", case=False)].sort_values("mean_folds", ascending=False)
SET_top10      = tag.sort_values("mean_folds", ascending=False).head(10)

SETS = {
    "best_per_modality":           SET_modality["sig"].tolist(),
    "best_per_classifier_anyFS":   SET_clf_anyfs["sig"].tolist(),
    "dietrich28_classifier_fixed": SET_dietrich28["sig"].tolist(),
    "top10":                       SET_top10["sig"].tolist(),
}
POOL = sorted({s for members in SETS.values() for s in members})

for name, d in [("best_per_modality", SET_modality),
                ("best_per_classifier_anyFS (descriptive: confounds clf with feature set)", SET_clf_anyfs),
                ("dietrich28_classifier_fixed (clean classifier RQ)", SET_dietrich28),
                ("top10", SET_top10)]:
    members = d["sig"].tolist()
    print("=" * 78)
    print(f"{name}:  {len(members)} experiments,  jointly-complete cities = {complete_cities(members)}")
    if len(d):
        print(d[["modality", "clf", "mean_folds", "n_cities", "sig"]].to_string(index=False))
print("=" * 78)
print(f"POOL (union for BCa CIs): {len(POOL)} experiments")

best_per_modality:  10 experiments,  jointly-complete cities = 8
            modality      clf  mean_folds  n_cities                                  sig
  MS_RGB_singlescene  XGBoost    0.748478        21                NB08b_BDA_RGB_XGBoost
SAR_coh_unsupervised AdaBoost    0.745562        12 NB08b_BDA_COH_DROP+RGB+CARD_AdaBoost
               other      GBM    0.741301        19                           CLF_F7_GBM
          MS_RGB+SAR AdaBoost    0.731546        21          NB08b_BDA_RGB+CARD_AdaBoost
      multimodal_all    other    0.681348        19              B7_UISEM_all_multimodal
             MS_only    other    0.653069        19                     B7_UISEM_ms_only
       SAR_card_only    other    0.631697        19                        R3b_card_only
        SAR_card+coh    other    0.620238        15              E6_B_nativeNaN_card+coh
        SAR_coh_only    other    0.566726        15              E6_B_nativeNaN_coh_only
    SAR_unsupervised    other    0.538353    

## Bootstrap BCa confidence intervals on mean-of-folds

Resamples cities (the fold unit). BCa where possible; falls back to percentile if BCa is degenerate. Computed over
the union (POOL) of all comparison sets - each experiment on its own cities, so no jointness is required.

In [4]:
def bca_ci(vec, n=2000, seed=42):
    v = np.asarray([x for x in vec if x==x], float)   # drop NaN
    if len(v) < 3 or np.allclose(v, v[0]):
        return (float(np.mean(v)) if len(v) else np.nan, np.nan, np.nan)
    try:
        r = bootstrap((v,), np.mean, confidence_level=0.95, n_resamples=n,
                      method="BCa", random_state=seed)
        return float(np.mean(v)), float(r.confidence_interval.low), float(r.confidence_interval.high)
    except Exception:
        r = bootstrap((v,), np.mean, confidence_level=0.95, n_resamples=n,
                      method="percentile", random_state=seed)
        return float(np.mean(v)), float(r.confidence_interval.low), float(r.confidence_interval.high)

def sets_of(sig):
    return "|".join(k for k, members in SETS.items() if sig in members)

ci_rows = []
for s in POOL:
    m, lo, hi = bca_ci(mat.loc[s].values)
    ci_rows.append({"experiment": s, "mean_folds": m, "ci95_low": lo, "ci95_high": hi,
                    "n_cities": int(np.sum(~np.isnan(mat.loc[s].values.astype(float)))),
                    "sets": sets_of(s)})
ci = pd.DataFrame(ci_rows).sort_values("mean_folds", ascending=False)
ci.to_csv(NB12 / "nb12b_meanfolds_bca_ci.csv", index=False)
pd.set_option("display.width", 220)
print(ci.to_string(index=False))
print("\nsaved:", NB12 / "nb12b_meanfolds_bca_ci.csv")

                          experiment  mean_folds  ci95_low  ci95_high  n_cities                                              sets
               NB08b_BDA_RGB_XGBoost    0.748478  0.721122   0.779828        21 best_per_modality|best_per_classifier_anyFS|top10
              NB08b_BDA_RGB_AdaBoost    0.748367  0.721054   0.780552        21                   best_per_classifier_anyFS|top10
NB08b_BDA_COH_DROP+RGB+CARD_AdaBoost    0.745562  0.706532   0.784997        12                           best_per_modality|top10
                NB08b_BDA_RGB_LogReg    0.744569  0.717688   0.778087        21                   best_per_classifier_anyFS|top10
     NB08b_BDA_COH_DROP+RGB_AdaBoost    0.744086  0.703049   0.785516        12                                             top10
            NB08b_BDA_RGB_ExtraTrees    0.743519  0.716884   0.775820        21                   best_per_classifier_anyFS|top10
                   NB08b_BDA_RGB_GBM    0.743394  0.717461   0.774847        21           

## Friedman omnibus + Wilcoxon posthoc (Holm) + Cliff's delta, per comparison set

Runs each set in `SETS` separately. Friedman tests whether the experiments in a set differ across their
jointly-complete cities; if there are enough complete cities, pairwise Wilcoxon signed-rank (paired by city, each
pair on its OWN complete cities) with Holm correction follows, plus Cliff's delta. Omnibus results across sets go to
`nb12b_friedman_omnibus.csv`; per-set pairwise tables to `nb12b_pairwise_wilcoxon_holm__<set>.csv`
(best_per_modality is also written to the canonical `nb12b_pairwise_wilcoxon_holm.csv` so 12d / downstream still
resolve).

In [5]:
omnibus_rows = []
MIN_COMPLETE = 3   # Friedman needs >=3 cities scored in EVERY member of the set

def pairwise_table(members):
    pairs, praw, deltas, npair = [], [], [], []
    for i in range(len(members)):
        for j in range(i + 1, len(members)):
            a, b = members[i], members[j]
            pair = mat.loc[[a, b]].dropna(axis=1, how="any")
            x, y = pair.loc[a].values, pair.loc[b].values
            if len(x) >= 3:
                try:
                    w, pp = wilcoxon(x, y, zero_method="wilcox")
                except Exception:
                    pp = np.nan
            else:
                pp = np.nan
            pairs.append((a, b)); praw.append(pp); deltas.append(cliffs_delta(x, y)); npair.append(int(len(x)))
    if not pairs:
        return None
    padj = holm([pp if pp == pp else 1.0 for pp in praw])
    return pd.DataFrame({"A": [a for a, _ in pairs], "B": [b for _, b in pairs],
                         "n_pair_cities": npair, "wilcoxon_p": praw, "p_holm": padj,
                         "cliffs_delta": deltas}).sort_values("p_holm")

for set_name, members in SETS.items():
    members = [m for m in members if m in mat.index]
    print("=" * 78)
    print(f"SET: {set_name}  ({len(members)} experiments)")
    if len(members) < 3:
        print("  skipped: need >=3 experiments")
        omnibus_rows.append({"set": set_name, "k": len(members), "complete_cities": np.nan,
                             "friedman_chi2": np.nan, "friedman_p": np.nan})
        continue

    sub = mat.loc[members]
    complete = sub.dropna(axis=1, how="any")
    nC = complete.shape[1]
    print(f"  jointly-complete cities: {nC}")
    if nC >= MIN_COMPLETE:
        stat, p = friedmanchisquare(*[complete.loc[s].values for s in members])
        print(f"  Friedman chi2={stat:.3f}  p={p:.4g}  (blocks=cities={nC}, k={len(members)})")
    else:
        stat, p = np.nan, np.nan
        print(f"  Friedman skipped (only {nC} jointly-complete cities; rely on BCa CIs + pairwise below)")
    omnibus_rows.append({"set": set_name, "k": len(members), "complete_cities": int(nC),
                         "friedman_chi2": (round(float(stat), 4) if stat == stat else np.nan),
                         "friedman_p": (round(float(p), 4) if p == p else np.nan)})

    pw = pairwise_table(members)
    if pw is not None:
        out = NB12 / f"nb12b_pairwise_wilcoxon_holm__{set_name}.csv"
        pw.to_csv(out, index=False)
        if set_name == "best_per_modality":
            pw.to_csv(NB12 / "nb12b_pairwise_wilcoxon_holm.csv", index=False)
        pd.set_option("display.width", 220)
        print(pw.to_string(index=False))
        print("  saved:", out)

friedman = pd.DataFrame(omnibus_rows)
friedman.to_csv(NB12 / "nb12b_friedman_omnibus.csv", index=False)
print("=" * 78)
print(friedman.to_string(index=False))
print("\nsaved:", NB12 / "nb12b_friedman_omnibus.csv")
print("|Cliff's delta|: <0.147 negligible, <0.33 small, <0.474 medium, else large")

SET: best_per_modality  (10 experiments)
  jointly-complete cities: 8
  Friedman chi2=49.745  p=1.203e-07  (blocks=cities=8, k=10)
                                   A                                    B  n_pair_cities  wilcoxon_p   p_holm  cliffs_delta
         NB08b_BDA_RGB+CARD_AdaBoost        B3a_LogRatio_VH_faithful_bldg             20    0.000002 0.000086      0.900000
               NB08b_BDA_RGB_XGBoost        B3a_LogRatio_VH_faithful_bldg             20    0.000004 0.000168      0.915000
                          CLF_F7_GBM        B3a_LogRatio_VH_faithful_bldg             19    0.000004 0.000168      0.911357
             B7_UISEM_all_multimodal        B3a_LogRatio_VH_faithful_bldg             19    0.000008 0.000320      0.795014
                    B7_UISEM_ms_only        B3a_LogRatio_VH_faithful_bldg             19    0.000038 0.001564      0.739612
                          CLF_F7_GBM                     B7_UISEM_ms_only             19    0.000210 0.008392      0.556787
 

## (optional) City-block bootstrap CI on pooled AUC

Heavier: reloads OOF and resamples whole cities, then pools their rows and recomputes pooled AUC. Set
`RUN_POOLED_CI = True` and a short `POOLED_SELECT` to run. Percentile CI.

In [6]:
RUN_POOLED_CI = False
POOLED_SELECT = SETS["best_per_modality"][:3]

if RUN_POOLED_CI:
    import pyarrow.parquet as pq
    from sklearn.metrics import roc_auc_score
    DRIVE_ROOT = Path(gs.DRIVE_ROOT)
    roots = [Path(getattr(gs,a)) for a in ["RESULTS_ROOT","OUTPUT_ROOT","DATA_OUTPUTS"] if getattr(gs,a,None)]
    roots += [DRIVE_ROOT/"data"/"outputs", RESULTS_ROOT]
    seen=set(); roots=[r for r in roots if r.exists() and (r not in seen and not seen.add(r))]
    def sig_of(n):
        s=re.sub(r"\.parquet$","",n); s=re.sub(r"^oof_","",s)
        return re.sub(r"__\d{8}_\d{6}_[0-9a-fA-F]+$","",s)
    allp=[]
    for r in roots: allp+=list(r.rglob("oof_*.parquet"))
    bysig={}
    for p in sorted(set(allp)):
        m=re.search(r"(\d{8}_\d{6})",p.name); bysig.setdefault(sig_of(p.name),[]).append((m.group(1) if m else "",p))
    def latest(sig): return sorted(bysig.get(sig,[]))[-1][1] if sig in bysig else None
    def pooled_auc(y,s):
        return float(roc_auc_score(y,s)) if len(np.unique(y))>1 else np.nan
    rng=np.random.default_rng(42); out=[]
    for sig in POOLED_SELECT:
        p=latest(sig)
        if p is None: print("no OOF for",sig); continue
        cc=[f.name for f in pq.ParquetFile(p).schema_arrow]
        d=pd.read_parquet(p, columns=[c for c in ["city","y_true","y_proba","is_final"] if c in cc])
        if "is_final" in d: d=d[~d["is_final"].astype(bool)]
        cities=d["city"].unique(); groups={c:d[d["city"]==c] for c in cities}
        base=pooled_auc(d["y_true"],d["y_proba"]); boot=[]
        for _ in range(1000):
            pick=rng.choice(cities,size=len(cities),replace=True)
            dd=pd.concat([groups[c] for c in pick]); boot.append(pooled_auc(dd["y_true"],dd["y_proba"]))
        lo,hi=np.nanpercentile(boot,[2.5,97.5])
        out.append({"experiment":sig,"pooled_auc":base,"ci95_low":float(lo),"ci95_high":float(hi)})
        print(f"{sig}: pooled={base:.4f}  CI[{lo:.4f},{hi:.4f}]")
    if out: pd.DataFrame(out).to_csv(NB12/"nb12b_pooled_cityblock_ci.csv", index=False)
else:
    print("pooled CI skipped (set RUN_POOLED_CI=True to run)")

pooled CI skipped (set RUN_POOLED_CI=True to run)


## How to report

- Quote **mean-of-folds with its BCa 95% CI** (`nb12b_meanfolds_bca_ci.csv`) as the headline cross-city number per
  experiment.
- For each research question, use the matching set's **Friedman + Holm** to claim a ranking is real; if Friedman is
  n.s. (or only a few cities are jointly complete), state the configurations are **not distinguishable** and report
  the pairwise Wilcoxon + Cliff's delta and the BCa CIs as the evidence.
- The clean "classifier choice does not matter after tuning" claim uses `dietrich28_classifier_fixed` (one feature
  set); `best_per_classifier_anyFS` is descriptive only (it confounds classifier with feature set).
- Put both pooled and mean-folds in the results table; never quote pooled alone.